In [36]:
import pandas as pd 
import numpy as np

In [37]:
df=pd.read_csv(r"IMDB Dataset.csv")

In [38]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [39]:
df["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [40]:
df.drop_duplicates(inplace=True)

In [41]:
df.shape

(49582, 2)

In [42]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [43]:
df["review"]=df["review"].str.lower()

In [44]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### Removing urls

In [45]:
import re

def remove_urls(text):
    op=re.sub(r"http\S+","",text)# sub("pattern","replace by","string")
    return op

df["review"]=df["review"].apply(remove_urls)

In [46]:
df.head(5)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [47]:
# removing Html tags

def remove_html(text):
     text=re.sub(r"<.*?>","",text)
     return text


df["review"]=df["review"].apply(remove_html)

In [48]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [49]:
#remove punctuations

def remove_punctuation(text):
    clear_text=re.sub("[^A-Za-z0-9\s]","",text)# expectA-Za-z0-9 and /s => for void spaces
    return clear_text


df["review"]=df["review"].apply(remove_punctuation)

In [50]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [51]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\91917\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\91917\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\91917\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [52]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

### Removing Stopwords

In [53]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    filtered_words=[]
    
    for word in tokens:
        if word not in stop_words:
           filtered_words.append(word)

    return " ".join(filtered_words)


df["review"]=df["review"].apply(remove_stopwords)        

In [54]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


## Stemming 

In [55]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps=PorterStemmer()
    tokenize=word_tokenize(text)
    stemmed_words=[]
    for word in tokenize:
        stem_op=ps.stem(word)
        stemmed_words.append(stem_op)

    return " ".join(stemmed_words)


df["review"]=df["review"].apply(stemming)

In [56]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product film techniqu unassum old...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


## Encoding

In [57]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()

df["sentiment"]=le.fit_transform(df["sentiment"])
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,1
1,wonder littl product film techniqu unassum old...,1
2,thought wonder way spend time hot summer weeke...,1
3,basic there famili littl boy jake think there ...,0
4,petter mattei love time money visual stun film...,1


In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer

tid=TfidfVectorizer(max_features=5000)

x=tid.fit_transform(df["review"])

In [59]:
x=x.toarray()

In [60]:
y=df["sentiment"]

## Train_Test_Splt


In [61]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42
)

In [62]:
X_test.shape

(9917, 5000)

In [63]:
y_train

7837     0
4814     0
35458    1
3446     0
24478    0
        ..
11304    0
45059    0
38405    0
860      1
15838    1
Name: sentiment, Length: 39665, dtype: int64

## Datasets DataLoader

In [87]:
import torch
from torch.utils.data import TensorDataset,DataLoader

train_Dataset=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)


test_Dataset=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [88]:
train_loader=DataLoader(train_Dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(test_Dataset,batch_size=64,shuffle=True)

## Build the RNN

In [67]:
import torch.nn as nn

In [89]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1): # num_layers=RNN (def=1)

        super(RNN,self).__init__()
        self.hidden_layer=hidden_size
        self.num_layer=num_layers

        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        #fc layer
        self.fc=nn.Linear(hidden_size,1)
        
    def forward(self,x):
        # optional h0(num_layers,batch_size,hidden_size)
        h0=torch.zeros(self.num_layer,x.shape[0],self.hidden_layer)

        op,_=self.rnn(x,h0)
        #1st value=> all hidden states of timestamps => (batch shape,seq_len(timestamp),hidden_size) 
        #2nd=> the previous hidden state at prev timestamp

        out=self.fc(op[:,-1,:])

        return out

In [90]:
import torch.optim as optim

input_size=X_train.shape[1]

model=RNN(input_size)

criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

## Train the model

In [92]:
epochs=10

train_losses=[]

for epoch in range(epochs):
    model.train()
    epoch_loss=0
    for Xb,yb in train_loader:
        optimizer.zero_grad()
        Xb=torch.unsqueeze(Xb,1)
        
        output=model(Xb)

        output=torch.sigmoid(output.squeeze()) #probability
        loss=criterion(output,yb)
        loss.backward()
        optimizer.step()

        epoch_loss+=loss.item()

    avg_train_loss=epoch_loss/len(train_loader)
    train_losses.append(avg_train_loss)



    print(f"{epoch+1}/{epochs} The Avg loss is: {avg_train_loss}")

1/10 The Avg loss is: 0.19813008912869037
2/10 The Avg loss is: 0.19740193976509957
3/10 The Avg loss is: 0.19629891381388712
4/10 The Avg loss is: 0.19516243534463068
5/10 The Avg loss is: 0.19507704230566178
6/10 The Avg loss is: 0.1942416726942024
7/10 The Avg loss is: 0.19356971419145985
8/10 The Avg loss is: 0.1934343092203621
9/10 The Avg loss is: 0.192367728240788
10/10 The Avg loss is: 0.19216610871015055


In [96]:
# Evaluation
corrected=0
total=0
model.eval()
with torch.no_grad():
    for Xb,yb in test_loader:
        Xb=Xb.unsqueeze(1)
        op=model(Xb)

        pred=(torch.sigmoid(op.squeeze())>0.5).float()

        corrected+=(pred==yb).sum().item()
        total+=yb.size(0)

    print(f"accuracy is :{(corrected/total)*100}")

accuracy is :86.94161540788545
